In [1]:
import numpy as np
import pandas as pd

## 1. Starting DataFrame

All examples use this simple but rich dataset:

In [2]:
df = pd.DataFrame({
    "city": ["NY", "NY", "NY", "LA", "LA", "LA"],
    "year": [2020, 2021, 2022, 2020, 2021, 2022],
    "sales": [100, 120, 130, 90, 95, 110]
})
df

,city,year,sales
0,NY,2020,100
1,NY,2021,120
2,NY,2022,130
3,LA,2020,90
4,LA,2021,95
5,LA,2022,110


## 2. Group → Aggregate → Unstack

### Mean sales per city per year (wide format)


In [3]:
g = df.groupby(["city", "year"])["sales"].mean()
g

city  year
LA    2020     90.0
      2021     95.0
      2022    110.0
NY    2020    100.0
      2021    120.0
      2022    130.0
Name: sales, dtype: float64

In [4]:
wide = g.unstack()
wide

year,2020,2021,2022
city,,,
LA,90.0,95.0,110.0
NY,100.0,120.0,130.0


#### Why this is useful:

Unstack turns a MultiIndex Series into a matrix, which makes differences and comparisons easy.


## 3. Compute Differences After Unstack

Year‑over‑year change per city


In [6]:
diff = wide.diff(axis=1)
diff

year,2020,2021,2022
city,,,
LA,NaN,5.0,15.0
NY,NaN,20.0,10.0


#### Key idea:

axis=1 means “compute differences across columns (years)”.

## 4. Stack Back to Long Format

Sometimes you compute in wide format but want tidy long format again.


In [7]:
long_diff = diff.stack()
long_diff

city  year
LA    2021     5.0
      2022    15.0
NY    2021    20.0
      2022    10.0
dtype: float64

## 5. Using Multiple Aggregations → Unstack

Mean and median sales per city per year

In [8]:
agg = df.groupby(["city", "year"])["sales"].agg(["mean", "median"])
agg

mean  median
city year               
LA   2020   90.0    90.0
     2021   95.0    95.0
     2022  110.0   110.0
NY   2020  100.0   100.0
     2021  120.0   120.0
     2022  130.0   130.0

In [9]:
wide_agg = agg.unstack()   # unstack year
wide_agg

mean               median              
year   2020   2021   2022   2020   2021   2022
city                                          
LA     90.0   95.0  110.0   90.0   95.0  110.0
NY    100.0  120.0  130.0  100.0  120.0  130.0

#### Key idea:

Multi-level columns appear when you aggregate multiple statistics.
Unstacking lets you pivot one of the index levels into columns.


## 6. Stack After Multi-Aggregation

You can also stack the columns back into a MultiIndex:


In [11]:
stacked = wide_agg.stack(future_stack=True)
stacked

mean  median
city year               
LA   2020   90.0    90.0
     2021   95.0    95.0
     2022  110.0   110.0
NY   2020  100.0   100.0
     2021  120.0   120.0
     2022  130.0   130.0

## 7. Group → Unstack → Compute Row Differences

Difference between cities for each year

In [12]:
wide_city = df.groupby(["year", "city"])["sales"].mean().unstack()
wide_city

city,LA,NY
year,,
2020,90.0,100.0
2021,95.0,120.0
2022,110.0,130.0


In [13]:
city_diff = wide_city["NY"] - wide_city["LA"]
city_diff

year
2020    10.0
2021    25.0
2022    20.0
dtype: float64

#### Key idea:

Unstacking lets you subtract one column from another directly.

## 8. Group → Stack → Compute Differences

Sometimes stacking first is cleaner.
Difference between years within each city (long-first approach)


In [14]:
s = df.groupby(["city", "year"])["sales"].mean()
s

city  year
LA    2020     90.0
      2021     95.0
      2022    110.0
NY    2020    100.0
      2021    120.0
      2022    130.0
Name: sales, dtype: float64

In [15]:
stacked = s.groupby(level=0).diff()   # diff within each city
stacked

city  year
LA    2020     NaN
      2021     5.0
      2022    15.0
NY    2020     NaN
      2021    20.0
      2022    10.0
Name: sales, dtype: float64

#### Key idea:

**groupby(level=0)** means “group by the first index level (city)”.

## 9. Group → Aggregate → Unstack → Compute Percent Change

Percent change year‑over‑year


In [16]:
pct = wide.pct_change(axis=1) * 100
pct

year,2020,2021,2022
city,,,
LA,NaN,5.555556,15.789474
NY,NaN,20.000000,8.333333


## 10. Group → Aggregate → Stack → Compute Rolling Stats

Rolling 2-year mean per city


In [17]:
s = df.groupby(["city", "year"])["sales"].mean()
s

city  year
LA    2020     90.0
      2021     95.0
      2022    110.0
NY    2020    100.0
      2021    120.0
      2022    130.0
Name: sales, dtype: float64

In [18]:
roll = s.groupby(level=0).rolling(2).mean()
roll

city  city  year
LA    LA    2020      NaN
            2021     92.5
            2022    102.5
NY    NY    2020      NaN
            2021    110.0
            2022    125.0
Name: sales, dtype: float64

Reshaping **before** aggregation is one of the most powerful (and least taught) patterns in pandas. It lets you reorganize the data first so that the later aggregation becomes simpler, more expressive, or more aligned with the question you’re answering.

Below is a structured set of examples that show **why** you would reshape first, **how** to do it, and **what kinds of statistical operations become easier** once the data is in the right shape.

## 1. Why reshape before aggregation?

Aggregating on the original layout often forces you into awkward multi-index structures.

Reshaping first lets you:
- pivot categories into columns so you can aggregate across them
- pivot time into columns so you can compute row-wise statistics
- stack columns into rows so you can group by a new variable
- create tidy long format so you can aggregate by a variable that didn’t exist before

This is the mental model:

**Reshape → Create new structure → Aggregate along the new structure**.

## 2. Example dataset

In [20]:
df = pd.DataFrame({
    "city": ["NY", "NY", "NY", "LA", "LA", "LA"],
    "year": [2020, 2021, 2022, 2020, 2021, 2022],
    "sales": [100, 120, 130, 90, 95, 110]
})

df

,city,year,sales
0,NY,2020,100
1,NY,2021,120
2,NY,2022,130
3,LA,2020,90
4,LA,2021,95
5,LA,2022,110


## 3. Unstack first → then aggregate across columns

**Goal: Compute the mean sales across years for each city**

Instead of grouping by city and year first, reshape so years become columns.
wide = df.pivot(index="city", columns="year", values="sales")
wide

In [21]:
wide = df.pivot(index="city", columns="year", values="sales")
wide

year,2020,2021,2022
city,,,
LA,90,95,110
NY,100,120,130


In [22]:
# Now aggregation becomes trivial:
wide.mean(axis=1)

city
LA     98.333333
NY    116.666667
dtype: float64

In [24]:
# sum sale totals across all cities
wide.sum(axis=0)

year
2020    190
2021    215
2022    240
dtype: int64

#### Why reshape first?

Because computing row-wise statistics across years is much easier when years are columns.

## 4. Unstack first → compute differences → then aggregate

**Goal: Compute the average year-over-year change per city**


In [25]:
wide = df.pivot(index="city", columns="year", values="sales")
wide

year,2020,2021,2022
city,,,
LA,90,95,110
NY,100,120,130


In [26]:
diff = wide.diff(axis=1)
diff

year,2020,2021,2022
city,,,
LA,NaN,5,15
NY,NaN,20,10


In [27]:
# Now aggregate:
diff.mean(axis=1)

city
LA    10.0
NY    15.0
dtype: float64

#### Why reshape first?

Because year-over-year differences require years to be columns.

## 5. Stack first → create a new variable → then aggregate

**Goal: Compute statistics by measurement type after stacking**

Let’s create a wide dataset with multiple numeric columns:


In [28]:
df2 = pd.DataFrame({
    "city": ["NY", "NY", "LA", "LA"],
    "sales": [100, 120, 90, 95],
    "profit": [30, 40, 25, 28]
})
df2

,city,sales,profit
0,NY,100,30
1,NY,120,40
2,LA,90,25
3,LA,95,28


In [29]:
# Stack the measurement columns:
stacked = df2.set_index("city").stack()
stacked

city        
NY    sales     100
      profit     30
      sales     120
      profit     40
LA    sales      90
      profit     25
      sales      95
      profit     28
dtype: int64

In [30]:
# Now you can aggregate by the new variable (“sales” vs “profit”)
stacked.groupby(level=1).mean()

profit     30.75
sales     101.25
dtype: float64

#### Why reshape first?

Stacking creates a new categorical variable (“sales” vs “profit”) that didn’t exist before.

## 6. Stack first → then compute statistics within each city

**Goal: Compute the range (max–min) of all numeric columns per city**


In [31]:
stacked = df2.set_index("city").stack()
stacked

city        
NY    sales     100
      profit     30
      sales     120
      profit     40
LA    sales      90
      profit     25
      sales      95
      profit     28
dtype: int64

In [32]:
stacked.groupby(level=0).apply(lambda s: s.max() - s.min())

city
LA    70
NY    90
dtype: int64

#### Why reshape first?

Stacking allows you to treat all numeric columns as one long vector per city.

## 7. Pivot first → then compute column-wise correlations

**Goal: Compute correlation between cities across years**

Reshape:


In [33]:
wide = df.pivot(index="year", columns="city", values="sales")
wide

city,LA,NY
year,,
2020,90,100
2021,95,120
2022,110,130


In [34]:
wide.corr()

city,LA,NY
city,,
LA,1.000000,0.891042
NY,0.891042,1.000000


#### Why reshape first?

Correlations require variables to be columns.

## 8. Melt first → then aggregate by a new variable

**Goal: Compute the mean across all numeric columns per city**

Start with df2:


In [36]:
df2

,city,sales,profit
0,NY,100,30
1,NY,120,40
2,LA,90,25
3,LA,95,28


In [37]:
long = df2.melt(id_vars="city", var_name="metric", value_name="value")
long

,city,metric,value
0,NY,sales,100
1,NY,sales,120
2,LA,sales,90
3,LA,sales,95
4,NY,profit,30
5,NY,profit,40
6,LA,profit,25
7,LA,profit,28


In [38]:
# Now aggregate:
long.groupby("city")["value"].mean()

city
LA    59.5
NY    72.5
Name: value, dtype: float64

In [39]:
long.groupby(["city", "metric"])["value"].mean()

city  metric
LA    profit     26.5
      sales      92.5
NY    profit     35.0
      sales     110.0
Name: value, dtype: float64

#### Why reshape first?
 Melt creates a tidy long format that allows aggregation across multiple columns.

## 9. Pivot first → then compute row-wise statistics

**Goal: Compute the median sales per city across years**


In [41]:
wide = df.pivot(index="city", columns="year", values="sales")
wide


year,2020,2021,2022
city,,,
LA,90,95,110
NY,100,120,130


In [42]:
wide.median(axis=1)

city
LA     95.0
NY    120.0
dtype: float64

#### Why reshape first?

Median across years is a row-wise operation.

## 10. Stack first → then compute z-scores within each city

**Goal: Standardize each city’s sales across years**


In [44]:
df

,city,year,sales
0,NY,2020,100
1,NY,2021,120
2,NY,2022,130
3,LA,2020,90
4,LA,2021,95
5,LA,2022,110


In [45]:
s = df.set_index(["city", "year"])["sales"]
s

city  year
NY    2020    100
      2021    120
      2022    130
LA    2020     90
      2021     95
      2022    110
Name: sales, dtype: int64

In [46]:
stacked = s.groupby(level=0).transform(lambda x: (x - x.mean()) / x.std())
stacked

city  year
NY    2020   -1.091089
      2021    0.218218
      2022    0.872872
LA    2020   -0.800641
      2021   -0.320256
      2022    1.120897
Name: sales, dtype: float64

#### Why reshape first?

Stacking creates a MultiIndex that makes grouping by city trivial.

## Closing thought

Reshaping before aggregation is about setting up the data so the aggregation becomes natural.
You’ve already mastered stack/unstack mechanically—this is the layer where they become strategic tools.
